# Decision Tree

This notebook trains and evaluates a baseline Decision Tree, then tunes the
tree using cross-validation on the training data only. A Decision Tree is a
rule-based classification algorithm that splits bookings into smaller groups
using feature thresholds, then predicts whether a booking is cancelled from
the final leaf node. The goal is to compare baseline and tuned performance
and check whether tuning reduces overfitting.

In [ ]:
from pathlib import Path
import sys

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "data/processed/hotel_booking_features.csv").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

from scripts.model_evaluation_utils import (
    RANDOM_STATE,
    OUTPUT_DIR,
    load_feature_dataset,
    split_feature_dataset,
    class_distribution,
    build_pipeline,
    get_cv,
    cross_validation_summary,
    fit_and_evaluate,
    confusion_matrix_frame,
)

FIGURE_DIR = OUTPUT_DIR / "Figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Shared Data Split

This section loads the same feature-engineered dataset and train/test split
used by the other model notebooks. The class distribution is displayed so
the tree results can be read with the cancellation imbalance in mind. Since
the imbalance is moderate rather than extreme, class weighting is not used as
the default Decision Tree setting.

In [ ]:
data = load_feature_dataset()
prepared = split_feature_dataset(data)

print(f"Training rows: {len(prepared.X_train):,}")
print(f"Test rows: {len(prepared.X_test):,}")

print("\nTraining class distribution")
display(class_distribution(prepared.y_train))

print("\nTest class distribution")
display(class_distribution(prepared.y_test))

## 2. Baseline Decision Tree

The baseline tree is created with default depth settings and without class
weighting. Because the cancelled class imbalance is moderate, the baseline
first learns from the natural class distribution instead of forcing extra
weight onto cancelled bookings. This gives an intentionally flexible model
so we can see how well a single tree fits the data before pruning or
limiting its complexity.

In [ ]:
tree_model = DecisionTreeClassifier(random_state=RANDOM_STATE)
tree_pipeline = build_pipeline(tree_model, prepared.X_train)
tree_pipeline

## 3. Baseline Cross-Validation

Cross-validation tests the baseline tree on repeated training folds. The
train-versus-validation gap is important here because unrestricted decision
trees often overfit by memorizing detailed patterns in the training data.

In [ ]:
tree_cv = cross_validation_summary(
    "Decision Tree",
    "Baseline",
    tree_pipeline,
    prepared.X_train,
    prepared.y_train,
)
display(tree_cv.round(4))

In [ ]:
baseline_gap = tree_cv.assign(
    train_cv_gap=tree_cv["train_mean"] - tree_cv["cv_mean"]
)
display(baseline_gap[["metric", "train_mean", "cv_mean", "train_cv_gap"]].round(4))

**Baseline interpretation:** A Decision Tree can capture non-linear patterns
and feature interactions, but an unrestricted tree can memorize the training
data. The train-vs-CV gap table is used to check overfitting. A large gap,
especially for F1, suggests that the baseline tree is too complex.

## 4. Baseline Final Test Evaluation

This evaluates the baseline tree on the untouched test set. The result
shows whether the high flexibility of the tree actually generalizes to new
bookings or mainly helped on the training folds.

In [ ]:
tree_pipeline, tree_test = fit_and_evaluate(
    "Decision Tree",
    "Baseline",
    tree_pipeline,
    prepared,
)

tree_test_results = pd.DataFrame([tree_test])
display(tree_test_results.round(4))

## 5. Baseline Confusion Matrix

The baseline confusion matrix shows the actual prediction mistakes. It
explains how many cancellations the tree missed and how many non-cancelled
bookings it incorrectly predicted as cancelled.

In [ ]:
tree_predictions = tree_pipeline.predict(prepared.X_test)
tree_confusion = confusion_matrix_frame(prepared.y_test, tree_predictions)
display(tree_confusion)

## 6. Decision Tree Tuning

The grid search uses F1-score because cancellation prediction should balance
precision and recall. Tuning is used to choose better tree complexity
settings instead of manually selecting them. For Decision Trees, tuning is
especially important because an unrestricted tree can become too deep and
memorize the training data. Tuning is performed with training data and
cross-validation only.

In [ ]:
tuned_tree_base = build_pipeline(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    prepared.X_train,
)

tree_param_grid = {
    "model__max_depth": [6, 10],
    "model__min_samples_split": [20, 100],
    "model__min_samples_leaf": [10, 25],
    "model__criterion": ["gini", "entropy"],
}

tree_search = GridSearchCV(
    estimator=tuned_tree_base,
    param_grid=tree_param_grid,
    scoring="f1",
    cv=get_cv(),
    n_jobs=-1,
    return_train_score=True,
)
tree_search.fit(prepared.X_train, prepared.y_train)

print("Best Decision Tree parameters:")
print(tree_search.best_params_)
print(f"Best CV F1: {tree_search.best_score_:.4f}")

**Tuning choice:** F1-score is used for tuning because cancellation
prediction needs a balance between precision and recall. False positives may
cause unnecessary follow-up actions, while false negatives mean the hotel may
miss bookings that are likely to cancel. `max_depth`, `min_samples_split`,
and `min_samples_leaf` control how complex the tree can become, so tuning
these values helps reduce overfitting. ROC-AUC and PR-AUC are still reported
to understand ranking and probability discrimination beyond one fixed
threshold.

## 7. Tuned Cross-Validation

The best tree from grid search is evaluated with the same cross-validation
summary. The goal is to check whether limiting depth and leaf sizes reduces
overfitting while keeping useful cancellation prediction performance.

In [ ]:
tuned_tree_pipeline = tree_search.best_estimator_

tuned_tree_cv = cross_validation_summary(
    "Decision Tree",
    "Tuned",
    tuned_tree_pipeline,
    prepared.X_train,
    prepared.y_train,
)
display(tuned_tree_cv.round(4))

In [ ]:
tuned_gap = tuned_tree_cv.assign(
    train_cv_gap=tuned_tree_cv["train_mean"] - tuned_tree_cv["cv_mean"]
)
display(tuned_gap[["metric", "train_mean", "cv_mean", "train_cv_gap"]].round(4))

**Tuned model interpretation:** The tuned tree should reduce the training
score compared with the unrestricted baseline. That is expected and usually
desirable when the baseline has a large train-vs-CV gap. The important
question is whether validation and test performance become more reliable.

## 8. Tuned Final Test Evaluation

This applies the tuned tree to the final test set. It shows whether the
simpler tree structure chosen during tuning gives more reliable performance
on unseen bookings.

In [ ]:
tuned_tree_pipeline, tuned_tree_test = fit_and_evaluate(
    "Decision Tree",
    "Tuned",
    tuned_tree_pipeline,
    prepared,
)

tuned_tree_test_results = pd.DataFrame([tuned_tree_test])
display(tuned_tree_test_results.round(4))

## 9. Tuned Confusion Matrix

The tuned confusion matrix shows how pruning-style constraints changed the
actual errors. It helps explain whether tuning reduced false positives,
false negatives, or both.

In [ ]:
tuned_tree_predictions = tuned_tree_pipeline.predict(prepared.X_test)
tuned_tree_confusion = confusion_matrix_frame(prepared.y_test, tuned_tree_predictions)
display(tuned_tree_confusion)

## 10. Baseline vs Tuned Comparison

This comparison places the unrestricted tree and tuned tree side by side.
It is used to decide whether reduced overfitting and better ranking metrics
are worth any change in default-threshold F1, precision or recall.

In [ ]:
decision_tree_test_results = pd.DataFrame([
    tree_test,
    tuned_tree_test,
])
decision_tree_cv_results = pd.concat([
    tree_cv,
    tuned_tree_cv,
], ignore_index=True)

display(decision_tree_test_results.round(4))
display(decision_tree_cv_results.round(4))

In [ ]:
f1_comparison = decision_tree_test_results.loc[
    :, ["version", "precision", "recall", "f1", "roc_auc", "pr_auc"]
]
display(f1_comparison.round(4))

**Comparison interpretation:** The tuned tree should not be described as
better on every metric unless the table supports that claim. If tuned F1 is
lower but ROC-AUC and PR-AUC are higher, the correct conclusion is that
tuning improved ranking/probability discrimination and reduced overfitting,
while the default-threshold F1 did not improve. This trade-off is important
for the final model comparison.

## 11. ROC and Precision-Recall Curves

These curves show how the tuned tree separates cancelled and non-cancelled
bookings across thresholds. Precision-recall is especially useful because
the cancellation class is smaller than the non-cancellation class.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_estimator(tuned_tree_pipeline, prepared.X_test, prepared.y_test, ax=axes[0])
PrecisionRecallDisplay.from_estimator(tuned_tree_pipeline, prepared.X_test, prepared.y_test, ax=axes[1])
axes[0].set_title("Tuned Decision Tree ROC Curve")
axes[1].set_title("Tuned Decision Tree Precision-Recall Curve")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "decision_tree_curves.png", dpi=300, bbox_inches="tight")
plt.show()

## 12. Tree Visualization

This visualizes the first levels of the tuned decision tree. Decision Trees
do not use coefficients like Logistic Regression; instead, their explanation
comes from split rules and feature importance. This plot helps make the
model explainable by showing the early split rules that guide the most
important prediction paths.

In [ ]:
fitted_tree = tuned_tree_pipeline.named_steps["model"]

plt.figure(figsize=(20, 8))
plot_tree(
    fitted_tree,
    max_depth=3,
    filled=True,
    rounded=True,
    class_names=["Not Cancelled", "Cancelled"],
)
plt.title("Tuned Decision Tree - First Three Levels")
plt.savefig(FIGURE_DIR / "decision_tree_visualization.png", dpi=300, bbox_inches="tight")
plt.show()

## 13. Interpretation Notes

The baseline Decision Tree is expected to achieve very high training scores
because it can grow deep and memorize patterns. The tuned tree is more
constrained, so its training score may be lower, but its validation gap and
probability-based metrics can be stronger. Since the class imbalance is only
moderate, the notebook does not make class weighting the main approach; it
focuses on controlling tree complexity through tuning. Keep both baseline
and tuned results for the final comparison and explain the precision-recall
trade-off clearly.